In [2]:
import warnings
from pathlib import Path

import pandas as pd
import anndata as ad
from dotenv import dotenv_values

from hm3d_utils.utils import download_file

warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
# Settuping up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

OUTDIR = DATADIR / "7_spatial" / "raw"
DOWNLOAD_DIR = OUTDIR / "Zhang"
DOWNLOAD_DIR.mkdir(exist_ok=True, parents=True) 

DATASETS = ['Zhuang-ABCA-1', 'Zhuang-ABCA-2', 'Zhuang-ABCA-3','Zhuang-ABCA-4',]

OUTFILE = OUTDIR / "Zhang_spatial.h5ad"

# Downloading files 

In [16]:
for ds in DATASETS:

    cell_meta_link = f"https://allen-brain-cell-atlas.s3-us-west-2.amazonaws.com/metadata/{ds}/20241115/views/cell_metadata_with_cluster_annotation.csv"
    adata_link = f"https://allen-brain-cell-atlas.s3-us-west-2.amazonaws.com/expression_matrices/{ds}/20230830/{ds}-raw.h5ad"

    cell_meta_download_fp = DOWNLOAD_DIR / f"{ds}_cell_meta.csv"
    adata_download_fp = DOWNLOAD_DIR / f"{ds}-raw.h5ad"

    download_file(url = adata_link, 
                output_path = adata_download_fp, 
                overwrite = False)

    download_file(url = cell_meta_link, 
                output_path = cell_meta_download_fp, 
                overwrite = False)

Zhuang-ABCA-1-raw.h5ad already exists.
Zhuang-ABCA-1_cell_meta.csv already exists.
Zhuang-ABCA-2-raw.h5ad already exists.
Zhuang-ABCA-2_cell_meta.csv already exists.
Zhuang-ABCA-3-raw.h5ad already exists.
Zhuang-ABCA-3_cell_meta.csv already exists.
Zhuang-ABCA-4-raw.h5ad already exists.
Zhuang-ABCA-4_cell_meta.csv already exists.


# Loading the data creating new adatas 

In [8]:
adata_list = []
for ds in DATASETS:

    cell_meta_download_fp = DOWNLOAD_DIR / f"{ds}_cell_meta.csv"
    adata_download_fp = DOWNLOAD_DIR / f"{ds}-raw.h5ad"

    adata = ad.read_h5ad(adata_download_fp)
    cmeta = pd.read_csv(cell_meta_download_fp)

    # Filter adata to cells we have metadata for
    adata = adata[cmeta.cell_label, :].copy()
    adata.obs = cmeta.set_index("cell_label")

    # Rename coords to ccf
    adata.obs = adata.obs.rename(columns={
        "x": "x_ccf",
        "y": "y_ccf",
        "z": "z_ccf",
    })

    # drop problematic column that creates errors when saving the data
    adata.obs = adata.obs.drop(columns = "high_quality_transfer")

    # put gene names into var names
    adata.var = adata.var.reset_index().set_index("gene_symbol")

    adata_list.append(adata)

# Concat and save

In [13]:
concat = ad.concat(
    adata_list,
    merge="first"
)

In [14]:
concat

AnnData object with n_obs × n_vars = 5822737 × 1122
    obs: 'brain_section_label', 'feature_matrix_label', 'donor_label', 'donor_genotype', 'donor_sex', 'cluster_alias', 'x_ccf', 'y_ccf', 'z_ccf', 'subclass_confidence_score', 'cluster_confidence_score', 'neurotransmitter', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter_color', 'class_color', 'subclass_color', 'supertype_color', 'cluster_color'
    var: 'gene_identifier'

In [ ]:
#concat.to_h5ad(OUTFILE)